## Dans ce notebook, on testera l'adaptabilité de l'AE et du DAE à un capteur inconnu :
1. protocole d'évaluation : entrainement uniquement sur le roulement "DE", et test uniquement sur le roulement "FE"
2. Comparaison Auto-Encodeur (AE) et Denoising Auto-Encodeur (DAE)

In [1]:
# Initialisation
#  echo: false
# Recharge automatiquement les modules modifiés (évite de redémarrer le kernel)
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, confusion_matrix
from keras.callbacks import EarlyStopping

# Cherche le dossier racine du projet (BDS_Fault_Detection) pour faire les imports depuis Sources/
dossier = Path().resolve()
for parent in (dossier, *dossier.parents):
    if "BDS_Fault_Detection" in parent.name:
        RACINE = parent
sys.path.insert(0, str(RACINE))

# Importation des fonctions du projet (dossier Sources/). Attention de l'ordre d'importation  !
from Sources import data, config, features, models


I0000 00:00:1795767772.090402    8010 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
def construire_fenetres_brutes(selection, canal="DE", duree=config.DUREE_FENETRE, recouvrement=0.5):
    SB, lignes = [], []
    for nom, info in selection.iterrows():
        s = data.charger(nom, canaux=(canal,))
        for i, val in enumerate(features.fenetrer(s[canal], s["fs"], duree=duree, recouvrement=recouvrement)):
            SB.append(val)
            lignes.append({"fichier": nom, "fenetre": i, "canal": canal,
                           "defaut": info["defaut"], "charge_hp": info["charge_hp"],
                           "rpm": s["rpm"], "fs": s["fs"],
                           "defaut_sain": "Sain" if info["defaut"] == "Sain" else "Défaut",
                           "diametre_mils": info["diametre_mils"],
                           "classe": "Sain" if info["defaut"] == "Sain" else info["defaut"] + str(info["diametre_mils"]).zfill(3)})
    return np.array(SB, dtype=np.float32), pd.DataFrame(lignes)

def score_mse(modele, X):
    Xr = modele.predict(X[..., None], verbose=0)[..., 0]
    return ((X - Xr) ** 2).mean(axis=1)

def ajouter_bruit(x, snr_db, seed=42):
    rng = np.random.default_rng(seed)
    p = np.mean(x ** 2)
    return x + np.sqrt(p / 10 ** (snr_db / 10)) * rng.standard_normal(x.shape)

# --- DE + FE ---
selection_DE = data.selectionner(sources=("12kDE", "Normal"), defauts=("IR","B","OR"),
                                 diametres=(7,14,21), positions_or=("6",), charges=(0,1,2,3),
                                 canaux={"12kDE": ("DE",), "Normal": ("DE",)})
selection_FE = data.selectionner(sources=("12kFE", "Normal"), defauts=("IR","B","OR"),
                                 diametres=(7,14,21), positions_or=("3","6","12"), charges=(0,1,2,3),
                                 canaux={"12kFE": ("FE",), "Normal": ("FE",)})
SB_DE, infos_DE = construire_fenetres_brutes(selection_DE, canal="DE")
SB_FE, infos_FE = construire_fenetres_brutes(selection_FE, canal="FE")

SB = np.concatenate([SB_DE, SB_FE], axis=0)
infos = pd.concat([infos_DE, infos_FE], ignore_index=True)

# print(SB.shape); print(infos.groupby(["canal", "defaut_sain"]).size())

In [3]:
canal_train, canal_test = "FE", "DE"

sain = (infos["defaut_sain"] == "Sain").to_numpy()
i_train = np.where(infos["canal"].to_numpy() == canal_train)[0]
i_test  = np.where(infos["canal"].to_numpy() == canal_test)[0]  

# --- Vérifications anti-fuite ---
assert (infos["canal"].to_numpy()[i_train] == canal_train).all(), "le train doit être 100 % DE"
assert (infos["canal"].to_numpy()[i_test] == canal_test).all(), "le test doit être 100 % FE"
# seuls les fichiers Normal_* (sains) apparaissent des deux côtés : c'est le transfert
# inter-capteurs ; aucun fichier défaut des deux côtés :
fichiers_commun = np.intersect1d(infos["fichier"].to_numpy()[i_train],
                                 infos["fichier"].to_numpy()[i_test])
assert set(fichiers_commun) <= set(infos.loc[sain, "fichier"].unique()), \
    "fuite : un fichier avec défaut est présent dans le train ET le test"

SBNorm = features.standardiser_par_fenetre(SB)
X_train_sain = SBNorm[i_train][sain[i_train]]  

# Affichage plan de test
pd.DataFrame({
    "train": infos.iloc[i_train[sain[i_train]]][["defaut_sain","canal"]].value_counts(),
    "test":  infos.iloc[i_test][["defaut_sain","canal"]].value_counts(),
}).fillna(0).astype(int)

train  test
defaut_sain canal             
Défaut      DE         0  3605
Sain        DE         0  1410
            FE      1410     0

In [4]:
def evaluer_par_capteur(modele, tag, canaux_test=(canal_test,)):
    score = score_mse(modele, SBNorm)
    seuil = np.percentile(score[i_train][sain[i_train]], 99)   # p99 des sains du train
    s_test, y = score[i_test], (infos["defaut_sain"].to_numpy()[i_test] == "Défaut").astype(int)
    c = infos["canal"].to_numpy()[i_test]
    lignes = []
    for canal in canaux_test:
        m = c == canal
        TN, FP, FN, TP = confusion_matrix(y[m], s_test[m] > seuil, labels=[0, 1]).ravel()
        lignes.append({"config": tag, "capteur": canal, "seuil_p99": round(float(seuil), 4),
                       "AUC": round(roc_auc_score(y[m], s_test[m]), 3),
                       "FPR": round(FP/(FP+TN), 4), "FNR": round(FN/(FN+TP), 4)})
    return pd.DataFrame(lignes)

modele_AE = models.conv1d_ae(SBNorm.shape[1])
modele_AE.fit(X_train_sain[..., None], X_train_sain[..., None], epochs=50, batch_size=64,
              validation_split=0.1, shuffle=True, verbose=1,
              callbacks=[EarlyStopping(monitor="val_loss", patience=5, min_delta=0.02, restore_best_weights=True)])
df_AE = evaluer_par_capteur(modele_AE, f"AE (train sains {canal_train} -> test {canal_test})")
# display(df_AE)

Epoch 1/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 13s 391ms/step - loss: 0.6662 - val_loss: 0.4116
Epoch 2/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 7s 331ms/step - loss: 0.2878 - val_loss: 0.2456
Epoch 3/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 11s 350ms/step - loss: 0.1798 - val_loss: 0.1579
Epoch 4/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 7s 345ms/step - loss: 0.0819 - val_loss: 0.0463
Epoch 5/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 7s 336ms/step - loss: 0.0348 - val_loss: 0.0285
Epoch 6/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 7s 331ms/step - loss: 0.0231 - val_loss: 0.0219
Epoch 7/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 7s 351ms/step - loss: 0.0190 - val_loss: 0.0193
Epoch 8/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 7s 346ms/step - loss: 0.0171 - val_loss: 0.0176
Epoch 9/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 8s 365ms/step - loss: 0.0166 - val_loss: 0.0175
Epoch 10/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 7s 357ms/step - loss: 0.0149 - val_loss: 0.0155
Epoch 11/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 7s 339ms/step - loss: 0.0137 - val_loss: 0.0146


In [5]:
# Data augmentation : on duplique le train sain bruité, cible = propre
X_aug, Y_aug = [X_train_sain], [X_train_sain]
for snr in [15, 10, 5, 0]:
    X_aug.append(ajouter_bruit(X_train_sain, snr, seed=snr))
    Y_aug.append(X_train_sain)
X_aug, Y_aug = np.concatenate(X_aug), np.concatenate(Y_aug)

modele_DAE = models.conv1d_ae(SBNorm.shape[1])
modele_DAE.fit(X_aug[..., None], Y_aug[..., None], epochs=50, batch_size=64,
               validation_split=0.1, shuffle=True, verbose=1,
               callbacks=[EarlyStopping(monitor="val_loss", patience=5, min_delta=0.02, restore_best_weights=True)])
df_DAE = evaluer_par_capteur(modele_DAE, f"DAE (train sains {canal_train} -> test {canal_test})")
# display(df_DAE)
display(pd.concat([df_AE, df_DAE]).set_index(["config", "capteur"]))

Epoch 1/50
100/100 ━━━━━━━━━━━━━━━━━━━━ 43s 354ms/step - loss: 0.3430 - val_loss: 0.3274
Epoch 2/50
100/100 ━━━━━━━━━━━━━━━━━━━━ 33s 334ms/step - loss: 0.0820 - val_loss: 0.2624
Epoch 3/50
100/100 ━━━━━━━━━━━━━━━━━━━━ 34s 338ms/step - loss: 0.0710 - val_loss: 0.2391
Epoch 4/50
100/100 ━━━━━━━━━━━━━━━━━━━━ 35s 346ms/step - loss: 0.0663 - val_loss: 0.2306
Epoch 5/50
100/100 ━━━━━━━━━━━━━━━━━━━━ 35s 342ms/step - loss: 0.0630 - val_loss: 0.2246
Epoch 6/50
100/100 ━━━━━━━━━━━━━━━━━━━━ 24s 239ms/step - loss: 0.0604 - val_loss: 0.2200
Epoch 7/50
100/100 ━━━━━━━━━━━━━━━━━━━━ 23s 229ms/step - loss: 0.0591 - val_loss: 0.2189
Epoch 8/50
100/100 ━━━━━━━━━━━━━━━━━━━━ 21s 213ms/step - loss: 0.0581 - val_loss: 0.2178
Epoch 9/50
100/100 ━━━━━━━━━━━━━━━━━━━━ 22s 222ms/step - loss: 0.0574 - val_loss: 0.2163
Epoch 10/50
100/100 ━━━━━━━━━━━━━━━━━━━━ 23s 231ms/step - loss: 0.0569 - val_loss: 0.2154
Epoch 11/50
100/100 ━━━━━━━━━━━━━━━━━━━━ 21s 213ms/step - loss: 0.0564 - val_loss: 0.2144
Epoch 12/50
100/100

,,seuil_p99,AUC,FPR,FNR
config,capteur,,,,
AE (train sains FE -> test DE),DE,0.0271,1.0,0.0035,0.0
DAE (train sains FE -> test DE),DE,0.0144,1.0,0.0014,0.0
